# MLP Image Classification Project

---

## Purpose

This notebook is a **self-directed project** to practice building an image classifier using a Multi-Layer Perceptron (MLP) in pure PyTorch. No fastai, no HuggingFace - just you and PyTorch.

**What you'll practice:**
- Custom Dataset and DataLoader creation
- MLP architecture design with `nn.Module`
- Complete training loop from scratch
- Validation, metrics, and learning curves
- Hyperparameter experimentation

**Why MLPs on images?**

MLPs are not the best architecture for images (CNNs are). But that's the point:
1. You'll internalize PyTorch fundamentals through struggle
2. You'll see the ceiling of MLPs firsthand
3. When you learn CNNs later, you'll understand *why* they matter

---

## Dataset Options

Below are image datasets available in `/opt/ml-datasets/`. Choose one (or more) to work with.

### Summary Table

| Dataset | Images | Classes | Dimensions | Features (flattened) | MLP Difficulty |
|---------|--------|---------|------------|---------------------|----------------|
| **CIFAR-10** | 60,000 | 10 | 32×32×3 | 3,072 | Medium |
| **CIFAR-100** | 60,000 | 100 | 32×32×3 | 3,072 | Hard |
| **Fashion-MNIST** | 70,000 | 10 | 28×28 | 784 | Easy |
| **Oxford Pets** | 7,393 | 37 | Variable | Resize to 64×64 = 12,288 | Medium-Hard |
| **Flowers102** | 8,189 | 102 | Variable | Resize to 64×64 = 12,288 | Hard |
| **Food101** | 101,000 | 101 | Variable | Resize to 64×64 = 12,288 | Hard |

---

## Dataset Deep Dive

### 1. CIFAR-10

**What it is:** 60,000 32×32 color images in 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). Classic deep learning benchmark from 2009.

**Location:** `/opt/ml-datasets/cifar10/`

| Pros | Cons |
|------|------|
| Classic benchmark - can compare your results to literature | Small 32×32 images lose detail |
| Works directly with torchvision - no preprocessing needed | Somewhat artificial/toy-like |
| Well-studied - lots of tutorials available | MLP will plateau around 55-60% |
| Good size (60k) for training | |

**Expected MLP accuracy:** ~55-60% (CNNs get 90%+)

---

### 2. CIFAR-100

**What it is:** Same as CIFAR-10 but with 100 fine-grained classes (e.g., different types of fish, flowers, vehicles). Also has 20 "superclasses".

**Location:** `/opt/ml-datasets/cifar100/`

| Pros | Cons |
|------|------|
| More challenging than CIFAR-10 | 100 classes is very hard for MLP |
| Same easy loading as CIFAR-10 | Small images make fine-grained hard |
| Can use superclasses (20) for easier task | May be frustrating if accuracy stays low |

**Expected MLP accuracy:** ~25-35% on 100 classes, ~45-55% on 20 superclasses

---

### 3. Fashion-MNIST

**What it is:** 70,000 28×28 grayscale images of clothing items (T-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot).

**Location:** `/opt/ml-datasets/fashion_mnist/`

| Pros | Cons |
|------|------|
| Drop-in MNIST replacement - familiar format | Very similar to MNIST you already did |
| Grayscale = simpler (784 features) | Might feel repetitive |
| Well-defined benchmark | Less "real-world" than photos |
| MLP can actually do reasonably well | |

**Expected MLP accuracy:** ~88-90%

---

### 4. Oxford Pets

**What it is:** 7,393 images of 37 pet breeds (25 dog breeds, 12 cat breeds). Real photographs with varied backgrounds, poses, and lighting.

**Location:** `/opt/ml-datasets/oxford_pets/`

| Pros | Cons |
|------|------|
| Real photographs - feels like actual ML work | Requires resizing (variable dimensions) |
| Interesting subject matter | Smaller dataset (7.4k images) |
| 37 classes is challenging but not impossible | Need to parse annotation files |
| Good preprocessing practice | Fine-grained (breed) classification is hard |

**Expected MLP accuracy:** ~15-30% (37 classes from real photos is hard for MLP)

---

### 5. Flowers102

**What it is:** 8,189 images of 102 flower species. Beautiful high-quality photographs.

**Location:** `/opt/ml-datasets/flowers102/`

| Pros | Cons |
|------|------|
| Beautiful images - satisfying to work with | 102 classes is very hard |
| Fine-grained classification challenge | Small dataset (8k) |
| Good for learning preprocessing | Labels in .mat format (needs scipy) |
| Real practical application | MLP will struggle significantly |

**Expected MLP accuracy:** ~5-15% (102 fine-grained classes is brutal for MLP)

---

### 6. Food101

**What it is:** 101,000 images of 101 food categories (pizza, sushi, hamburger, etc.). 1,000 images per class.

**Location:** `/opt/ml-datasets/food101/`

| Pros | Cons |
|------|------|
| Large dataset (101k) - plenty of data | 101 classes is hard |
| Practical application (food recognition) | Large = slower training |
| Well-balanced (1k per class) | Requires significant resizing |
| Real photographs | MLP won't perform well |

**Expected MLP accuracy:** ~10-20%

---

## Recommendations

Based on **maximizing learning** while keeping the project achievable:

### Top Pick: Oxford Pets

**Why:**
- Real photographs force you to deal with preprocessing (resize, normalize)
- 37 classes is hard enough to be interesting, not so hard as to be demoralizing
- You'll build a complete custom Dataset class (can't just use torchvision)
- Cats and dogs are more engaging than abstract categories
- Small enough to iterate quickly on your GPU

**Challenge level:** Medium-Hard

---

### Alternative 1: CIFAR-10

**Why:**
- Zero preprocessing needed - focus purely on model/training
- Classic benchmark - you can compare to published results
- MLP can get 55%+ which feels like progress (vs 10% baseline)
- Good if you want to spend less time on data loading

**Challenge level:** Medium

---

### Alternative 2: Flowers102 (for the ambitious)

**Why:**
- 102 fine-grained classes is genuinely hard
- Beautiful dataset - motivation matters
- You'll really see MLP limitations
- Great "before/after" when you redo with CNNs

**Challenge level:** Hard

---

### What I'd Skip

| Dataset | Why Skip |
|---------|----------|
| Fashion-MNIST | Too similar to MNIST you already did |
| Food101 | Too large, 101 classes too hard, slow iteration |
| CIFAR-100 | 100 classes with tiny images = frustration |

---

## Your Choice

**Dataset I'm using for this project:** CIFAR-10

---

## Project Tasks

Below are the tasks you need to complete. The order follows a logical flow: understand your data first, then preprocess, then build the PyTorch pipeline, then model and train.

> **Note:** The exact steps depend on your data format. Notes are provided for common scenarios.

---

### Phase 1: Data Exploration

*Goal: Understand what you're working with before writing any preprocessing code.*

**Task 1.1: Understand the data structure**
- Navigate to your dataset folder
- How are files organized?

| Format | Examples | How to identify |
|--------|----------|-----------------|
| Folder per class | `train/dog/img1.jpg` | Subdirectories with class names |
| Binary/pickle | `data_batch_1`, `.pkl` | No file extension, or pickle |
| CSV + images | `labels.csv` + `images/` folder | CSV with filename and label columns |
| Pre-built | torchvision datasets | No local files needed |

**Task 1.2: Load raw data**
- Load a sample of the data (don't preprocess yet)

| Format | Loading approach |
|--------|------------------|
| Folder per class | `PIL.Image.open()` or wait for `ImageFolder` |
| Binary/pickle | `pickle.load()` with `encoding='bytes'` |
| CSV + images | `pandas.read_csv()` + `PIL.Image.open()` |
| Pre-built | `torchvision.datasets.CIFAR10(...)` etc. |

**Task 1.3: Visualize samples**
- Display several images with their labels
- Confirm data loaded correctly
- Get a feel for the task difficulty

**Task 1.4: Inspect data properties**
- Dimensions: Are all images the same size?
- Dtype: uint8? float32?
- Value range: 0-255? 0-1? Something else?
- Class distribution: Balanced or imbalanced?
- Total samples: How much data do you have?

---

### Phase 2: Preprocessing

*Goal: Clean and transform data into a consistent format suitable for training.*

**Task 2.1: Resize (if needed)**
- Only needed for variable-size images (real photos, etc.)
- Fixed-size datasets (CIFAR, MNIST) can skip this
- Consider: larger = more detail but slower training

**Task 2.2: Normalize pixel values**
- Neural networks prefer small, centered values
- Common approaches:

| Method | Formula | When to use |
|--------|---------|-------------|
| Scale to 0-1 | `x / 255.0` | Simple, works well |
| Standardize | `(x - mean) / std` | When using pretrained models |

- Always convert to float32 (uint8 can't represent decimals)

**Task 2.3: Train/validation split**
- Hold out 10-20% for validation
- Use `sklearn.model_selection.train_test_split()` for numpy arrays
- Use `torch.utils.data.random_split()` for Dataset objects

> **Note:** Some datasets come pre-split (separate train/test folders or files). Use the provided split, but you may still want to carve out validation from train.

**Task 2.4: Handle any data issues**
- Missing values, corrupted images, mislabeled data
- Most benchmark datasets are clean; real-world data often isn't

---

### Phase 3: PyTorch Data Pipeline

*Goal: Get data into PyTorch's DataLoader format for efficient batched training.*

**Task 3.1: Convert to tensors**
- Images: `torch.float32`
- Labels: `torch.long` (required by CrossEntropyLoss)

| From | To tensor |
|------|-----------|
| Numpy array | `torch.tensor(arr, dtype=...)` or `torch.from_numpy(arr)` |
| PIL Image | `transforms.ToTensor()` |

**Task 3.2: Create Dataset**

| Scenario | What to use |
|----------|-------------|
| Already have tensors | `TensorDataset(data_tensor, labels_tensor)` |
| Folder per class | `ImageFolder(root, transform=...)` |
| Complex/custom loading | Subclass `Dataset` with `__init__`, `__len__`, `__getitem__` |

**Task 3.3: Create DataLoaders**
```python
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
```
- `shuffle=True` for training (reduces overfitting)
- `shuffle=False` for validation (reproducibility)
- Batch size: 32-128 typical; larger = more memory, potentially worse generalization

**Task 3.4: (Optional) Data augmentation**
- Random horizontal flips, rotations, color jitter
- Only apply to training data
- More useful for CNNs than MLPs (MLPs don't understand spatial transformations)
- Can be added via `transforms.Compose()` in Dataset

**Task 3.5: Verify your pipeline**
- Grab one batch: `images, labels = next(iter(train_loader))`
- Check shapes: `images.shape` should be `(batch_size, features)` for MLP
- Check dtypes: float32 for images, long for labels

---

### Phase 4: Model Architecture

*Goal: Define an MLP that takes your input size and outputs class predictions.*

**Task 4.1: Define your MLP**
- Subclass `nn.Module`
- Implement `__init__` (define layers) and `forward` (define computation)
- Start simple: Input → Hidden → Output

**Task 4.2: Choose layer sizes**
- **Input:** flattened image size (e.g., 3072 for CIFAR-10, 784 for MNIST)
- **Hidden:** start with 256-512, experiment later
- **Output:** number of classes (10 for CIFAR-10)
- Architecture patterns to try:

| Pattern | Example | Notes |
|---------|---------|-------|
| Constant width | 512 → 512 → 512 | Simple, good starting point |
| Pyramidal | 1024 → 512 → 256 | Gradual compression, often works well |
| Wide first layer | 2048 → 512 → 256 | More capacity to capture input patterns |

**Task 4.3: Layer ordering**
The standard order for each "block" in your network:
```
Linear → BatchNorm → Activation → Dropout
```
- **Linear**: transforms features
- **BatchNorm**: normalizes outputs (stabilizes training)
- **Activation**: introduces non-linearity
- **Dropout**: regularization (randomly zeros neurons)

**Task 4.4: Choose components**

| Component | Options | Notes |
|-----------|---------|-------|
| **Activation** | `nn.ReLU()` | Standard default |
| | `nn.LeakyReLU(0.1)` | Avoids "dead neurons" problem |
| **Normalization** | `nn.BatchNorm1d(features)` | Stabilizes training, slight regularization |
| **Regularization** | `nn.Dropout(0.2-0.5)` | Prevents overfitting; use lower values (0.2) if also using BatchNorm |

**Task 4.5: Weight initialization (optional)**
- PyTorch defaults are usually fine
- For ReLU/LeakyReLU networks, He initialization can help:
```python
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)
model.apply(init_weights)
```

**Task 4.6: Verify your model**
- Create model instance
- Pass a dummy batch: `model(torch.randn(64, input_size))`
- Check output shape: should be `(64, num_classes)`
- Print parameter count: `sum(p.numel() for p in model.parameters())`

---

### Phase 5: Training Loop

*Goal: Train the model and monitor progress.*

**Task 5.1: Setup**

| Component | Options | Notes |
|-----------|---------|-------|
| **Loss** | `nn.CrossEntropyLoss()` | Standard for classification |
| **Optimizer** | `Adam(model.parameters(), lr=1e-3)` | Good default, adaptive learning rates |
| | `SGD(model.parameters(), lr=0.01, momentum=0.9)` | Often better for images, needs tuning |
| **Weight decay** | `optimizer = Adam(..., weight_decay=1e-4)` | L2 regularization, helps prevent overfitting |

```python
# Simple default
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# More tuned (often better for image tasks)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
```

> **Reminder:** Move model and data to GPU:
> ```python
> model = model.to(device)
> # In training loop:
> images, labels = images.to(device), labels.to(device)
> ```

**Task 5.2: Training loop**
```python
model.train()  # Enable dropout, batchnorm training mode
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()      # Clear old gradients
    outputs = model(images)    # Forward pass
    loss = criterion(outputs, labels)
    loss.backward()            # Compute gradients
    optimizer.step()           # Update weights
```

**Task 5.3: Validation loop**
```python
model.eval()  # Disable dropout, use stored batchnorm stats
with torch.no_grad():  # Don't track gradients (saves memory)
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # compute loss and accuracy
```

**Task 5.4: Track progress**
- Store history in lists for plotting:
```python
history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
```
- Print progress every N epochs to monitor training

---

### Phase 6: Evaluation and Analysis

*Goal: Understand how well your model performs and where it fails.*

**Task 6.1: Plot learning curves**
- Training loss vs. epochs
- Validation loss vs. epochs
- Gap between them indicates overfitting:
  - Train loss ↓, val loss ↓ = learning
  - Train loss ↓, val loss ↑ = overfitting

**Task 6.2: Final test evaluation**
- Load the held-out test set (if separate from validation)
- Report final accuracy — this is your "true" performance

**Task 6.3: Confusion matrix**
- Which classes get confused with which?
- Use `sklearn.metrics.confusion_matrix` + `seaborn.heatmap`
- Look for patterns (e.g., cat↔dog, bird↔airplane)

**Task 6.4: Per-class accuracy**
- Which classes are easiest/hardest?
- Helps identify where the model struggles

**Task 6.5: Visualize predictions**
- Show correctly classified examples
- Show misclassified examples with predicted vs actual labels
- Ask: Can YOU tell what these images are? (often hard at 32x32)

**Task 6.6: Understand limitations**
For MLPs on images, common failure patterns:
- Similar colors/textures get confused (cat↔dog)
- Similar backgrounds get confused (bird↔airplane in blue sky)
- MLPs learn color statistics, not shapes

---

### Phase 7: Experimentation

*Goal: Improve your results through systematic experimentation.*

**Task 7.1: Architecture tuning**

| What to try | Options |
|-------------|---------|
| Width | 256, 512, 1024, 2048 |
| Depth | 2, 3, 4 hidden layers |
| Pattern | Constant, pyramidal, wide-first |
| Activation | ReLU, LeakyReLU |
| BatchNorm | With/without |

**Task 7.2: Training tuning**

| What to try | Options |
|-------------|---------|
| Learning rate | 1e-2, 1e-3, 1e-4 |
| Optimizer | Adam, SGD+momentum |
| Weight decay | 0, 1e-4, 1e-3 |
| Dropout | 0.2, 0.3, 0.5 |
| Batch size | 32, 64, 128 |

**Task 7.3: Learning rate scheduling**
```python
# Reduce LR when validation plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5, factor=0.5
)
scheduler.step(val_accuracy)

# Smooth cosine decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
scheduler.step()  # Call after each epoch
```

**Task 7.4: Early stopping**
Stop training when validation performance stops improving:
```python
best_val_acc = 0
patience_counter = 0
patience = 10

for epoch in range(num_epochs):
    # ... training ...
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping")
            break

# Load best model
model.load_state_dict(torch.load('best_model.pth'))
```

**Task 7.5: Document your findings**
- What hyperparameters worked best?
- What was your best accuracy?
- Where did the MLP struggle? (Which classes? Why?)
- What would a CNN do differently?

---

### Key Takeaways

**What MLPs CAN learn from images:**
- Color statistics ("lots of blue = probably sky = airplane or bird")
- Texture patterns ("brown fuzzy = probably animal")

**What MLPs CANNOT learn:**
- Spatial patterns (edges, shapes, object parts)
- Position invariance (dog in corner vs center = different numbers)

**Expected MLP ceiling on CIFAR-10:** ~55-60%
**CNN performance on CIFAR-10:** ~90%+

*The gap shows why architecture matters more than hyperparameters.*

---

*Updated: 2025_01_12_1430*

## Hints (Only Read If Stuck)

<details>
<summary>Hint 1: Loading Oxford Pets images</summary>

Images are in `/opt/ml-datasets/oxford_pets/images/`. The filename format is `Breed_Name_123.jpg`. The breed is everything before the last underscore-number. There's also a `trainval.txt` and `test.txt` in the annotations folder.

</details>

<details>
<summary>Hint 2: Loading CIFAR-10</summary>

Use `torchvision.datasets.CIFAR10(root='/opt/ml-datasets/cifar10', train=True, download=False)`. The images are already tensors.

</details>

<details>
<summary>Hint 3: Flattening in forward pass</summary>

In your model's `forward` method, use `x = x.view(x.size(0), -1)` to flatten the input while preserving batch dimension.

</details>

<details>
<summary>Hint 4: Common training loop structure</summary>

```
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        # forward, loss, backward, step
    
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            # forward, compute metrics
```

</details>

<details>
<summary>Hint 5: GPU usage</summary>

Move model and data to GPU: `model.to('cuda')`, `images.to('cuda')`, `labels.to('cuda')`. Check availability with `torch.cuda.is_available()`.

</details>

---

## Reference Materials

**PyTorch documentation:**
- [torch.utils.data.Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)
- [torch.nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [torch.optim](https://pytorch.org/docs/stable/optim.html)

**Your previous notebooks:**
- `01_neural_network_part_1.ipynb` - How neural networks work
- `02_neural_network_part_2_training.ipynb` - Training mechanics

---

## Your Workspace

Start coding below. Good luck!

---

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


### Progress bar

In [ ]:
# 1 - Load the data somehow, e.g in this case cifar10 is binary, so we load the train into one np array
# Then we load the labels into another np array.

# 2. Normalize from 0-255 to 0 - 1

# 3. Split into train and validation using sklearn
# Could use pure pytorch, but common to use sklearn if numpy is involved. It will depend on how
# the data is loaded

In [ ]:
# Your imports here


In [ ]:
# Phase 1: Data Loading

import pickle
import numpy as np

# Load one batch file
with open('/opt/ml-datasets/cifar10/cifar-10-batches-py/data_batch_1', 'rb') as file:
    batch = pickle.load(file, encoding='bytes')

# Inspect what we got
print("Keys:", batch.keys())
print("Data shape:", batch[b'data'].shape)
print("Number of labels:", len(batch[b'labels']))

In [ ]:
# Combine all 5 training batches into one
all_data = []
all_labels = []

for i in range(1, 6):
    with open(f'/opt/ml-datasets/cifar10/cifar-10-batches-py/data_batch_{i}', 'rb') as file:
        batch = pickle.load(file, encoding='bytes')
        all_data.append(batch[b'data'])
        all_labels.extend(batch[b'labels'])

train_data = np.concatenate(all_data, axis=0)
train_labels = np.array(all_labels)

print(f"Combined training data: {train_data.shape}")
print(f"Combined training labels: {train_labels.shape}")

In [ ]:
# Visualize a few samples
import matplotlib.pyplot as plt

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
            'dog', 'frog', 'horse', 'ship', 'truck']

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    # Get one flattened image
    flat_image = train_data[i]

    # Reshape: (3072,) -> (3, 32, 32) -> (32, 32, 3)
    image = flat_image.reshape(3, 32, 32).transpose(1, 2, 0)

    ax.imshow(image)
    ax.set_title(class_names[train_labels[i]])
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
train_data[0:10]

In [ ]:
train_data = train_data.astype(np.float32) / 255.0

In [ ]:
print(train_data[0].tolist())  # Converts to Python list, prints all

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data, train_labels, val_labels = train_test_split(
    train_data, train_labels, test_size=0.2, random_state=42
)

In [ ]:
train_data.shape
val_data.shape

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import TensorDataset


In [ ]:
train_data_tensor = torch.tensor(train_data, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)
val_data_tensor = torch.tensor(val_data, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)

In [ ]:
train_data_tensor.device

In [ ]:
train_dataset = TensorDataset(train_data_tensor, train_labels_tensor)
val_dataset = TensorDataset(val_data_tensor, val_labels_tensor)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False) # no need to shuffle for val

In [ ]:
images, labels = next(iter(train_dataloader))

print(images[0].squeeze())

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
# Phase 3: Model Architecture

class BasicMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(in_features=3072, out_features=512),
            nn.ReLU(),
            nn.Linear(in_features=512, out_features=512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
model = BasicMLP()
dummy_batch = torch.randn(64, 3072)  # fake batch
output = model(dummy_batch)
print(output.shape)  # should be (64, 10)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
model = BasicMLP().to(device)

### Training loop - Pre improvements

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = len(dataloader)

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() # logging

    # Return average training loss for this epoch
    return total_loss / num_batches

def test_loop(dataloader, model, loss_fn, device):
    """
    We disable dropout and tracking of gradients and essentially
    just test running forward pass on our validation data
    """
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy = correct / size

    # Return loss and accuracy
    return test_loss, accuracy


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = BasicMLP().to(device)

loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Track history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

num_epochs = 50
for epoch in range(num_epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, device)
    val_loss, val_accuracy = test_loop(val_dataloader, model, loss_fn, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)

    # Print progress every 10 epochs (less noisy)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {100*val_accuracy:.1f}%")

# Print summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total epochs: {num_epochs}")
print(f"Best validation accuracy: {100*max(history['val_accuracy']):.1f}% (epoch {history['val_accuracy'].index(max(history['val_accuracy']))+1})")
print(f"Final validation accuracy: {100*history['val_accuracy'][-1]:.1f}%")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

### Analysis

### Training loop - try to make improvements

● Options to experiment with:

  
  ### 1. Adam optimizer - often faster convergence than SGD
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
  
  ### 2. Different hidden sizes - wider or narrower
  nn.Linear(3072, 1024),  # wider
  
  or
  nn.Linear(3072, 256),   # narrower
  
  ### 3. Deeper network - more layers
  nn.Linear(3072, 512), nn.ReLU(), nn.Dropout(0.3),
  nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
  nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
  nn.Linear(128, 10)
  
  ### 4. Learning rate scheduler - reduce LR when stuck
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
  
  In training loop after test_loop:
  scheduler.step(val_loss)

In [ ]:
# Phase 3: Model Architecture

class BasicMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
  nn.Linear(3072, 1024), nn.ReLU(), nn.Dropout(0.3),
  nn.Linear(1024, 10)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()
    total_loss = 0
    num_batches = len(dataloader)

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Return average training loss for this epoch
    return total_loss / num_batches

def test_loop(dataloader, model, loss_fn, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy = correct / size

    # Return loss and accuracy
    return test_loss, accuracy


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = BasicMLP().to(device)

loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.01
# optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


# Track history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

num_epochs = 80
for epoch in range(num_epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, device)
    val_loss, val_accuracy = test_loop(val_dataloader, model, loss_fn, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)

    # Print progress every 10 epochs (less noisy)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {100*val_accuracy:.1f}%")

# Print summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total epochs: {num_epochs}")
print(f"Best validation accuracy: {100*max(history['val_accuracy']):.1f}% (epoch {history['val_accuracy'].index(max(history['val_accuracy']))+1})")
print(f"Final validation accuracy: {100*history['val_accuracy'][-1]:.1f}%")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

In [ ]:
# Model Architecture with BatchNorm1d
# BatchNorm normalizes the output of each layer, stabilizing and accelerating training
# Place it AFTER the linear layer, BEFORE the activation function

class MLPWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3072, 512),
            nn.BatchNorm1d(512),  # normalizes 512 features
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),  # normalizes 512 features
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 10)   # no batchnorm before output
        )
        
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# Test it
model_bn = MLPWithBatchNorm().to(device)
print(f"Parameters: {sum(p.numel() for p in model_bn.parameters()):,}")

# Note: BatchNorm adds a small number of parameters (2 per feature: gamma and beta)
# It also tracks running mean/variance for inference (model.eval() uses these)

In [ ]:
# Training with BatchNorm model

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = MLPWithBatchNorm().to(device)

loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Track history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

num_epochs = 80
for epoch in range(num_epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, device)
    val_loss, val_accuracy = test_loop(val_dataloader, model, loss_fn, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {100*val_accuracy:.1f}%")

print("\n" + "="*60)
print("TRAINING SUMMARY (with BatchNorm)")
print("="*60)
print(f"Total epochs: {num_epochs}")
print(f"Best validation accuracy: {100*max(history['val_accuracy']):.1f}% (epoch {history['val_accuracy'].index(max(history['val_accuracy']))+1})")
print(f"Final validation accuracy: {100*history['val_accuracy'][-1]:.1f}%")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

In [ ]:
# Optimized MLP - combining best techniques
# 1. Wider first layer (more capacity to capture input patterns)
# 2. BatchNorm for stable training
# 3. Lower dropout (0.2) since BatchNorm already regularizes
# 4. SGD with momentum for smoother optimization
# 5. Learning rate scheduler to fine-tune when plateauing

class OptimizedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # Wider first layer - captures more input patterns
            nn.Linear(3072, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2),  # lower dropout with batchnorm
            
            # Pyramidal structure - gradual compression
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.flatten(x)
        return self.linear_relu_stack(x)

model = OptimizedMLP().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training with optimized setup

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = OptimizedMLP().to(device)

loss_fn = nn.CrossEntropyLoss()

# SGD with momentum - smoother optimization
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Learning rate scheduler - reduce LR by half when val accuracy plateaus for 5 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

# Track history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

# Early stopping setup
best_val_acc = 0
patience = 15
patience_counter = 0
best_model_state = None

num_epochs = 100
for epoch in range(num_epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, device)
    val_loss, val_accuracy = test_loop(val_dataloader, model, loss_fn, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    
    # Step the scheduler based on validation accuracy
    scheduler.step(val_accuracy)
    
    # Early stopping check
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {100*val_accuracy:.1f}% | LR: {current_lr:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\n" + "="*60)
print("TRAINING SUMMARY (Optimized MLP)")
print("="*60)
print(f"Total epochs run: {len(history['train_loss'])}")
print(f"Best validation accuracy: {100*max(history['val_accuracy']):.1f}% (epoch {history['val_accuracy'].index(max(history['val_accuracy']))+1})")
print(f"Final model loaded from best epoch")

In [ ]:
# Maximum effort MLP - pushing the limits
# Adding: wider layers, LeakyReLU, He initialization, weight decay, cosine annealing

class MaxEffortMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_stack = nn.Sequential(
            # Very wide first layer - maximum pattern capture
            nn.Linear(3072, 2048),
            nn.BatchNorm1d(2048),
            nn.LeakyReLU(0.1),  # avoids dead neurons
            nn.Dropout(0.2),
            
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            
            nn.Linear(512, 10)
        )
        
        # He initialization for LeakyReLU
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        
    def forward(self, x):
        x = self.flatten(x)
        return self.linear_stack(x)

model = MaxEffortMLP().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training with maximum effort setup

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = MaxEffortMLP().to(device)

loss_fn = nn.CrossEntropyLoss()

# SGD with momentum AND weight decay (L2 regularization)
optimizer = torch.optim.SGD(
    model.parameters(), 
    lr=0.01, 
    momentum=0.9, 
    weight_decay=1e-4  # L2 regularization
)

# Cosine annealing - smoothly reduces LR following cosine curve
num_epochs = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Track history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

# Early stopping
best_val_acc = 0
patience = 20
patience_counter = 0
best_model_state = None

for epoch in range(num_epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, device)
    val_loss, val_accuracy = test_loop(val_dataloader, model, loss_fn, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    
    # Step cosine scheduler (every epoch)
    scheduler.step()
    
    # Early stopping check
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {100*val_accuracy:.1f}% | LR: {current_lr:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\n" + "="*60)
print("TRAINING SUMMARY (Maximum Effort MLP)")
print("="*60)
print(f"Total epochs run: {len(history['train_loss'])}")
print(f"Best validation accuracy: {100*max(history['val_accuracy']):.1f}% (epoch {history['val_accuracy'].index(max(history['val_accuracy']))+1})")
print(f"Final model loaded from best epoch")

## Evaluation & Analysis

Understanding *why* the MLP struggles is more valuable than the accuracy number itself. This section teaches you to diagnose model failures.

In [ ]:
# 4. Learning Curves - Visualize Training Progress

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Training Loss', color='blue')
axes[0].plot(history['val_loss'], label='Validation Loss', color='orange')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# The gap between train and val loss indicates overfitting
# If they diverge, the model is memorizing rather than learning

# # Accuracy curve
axes[1].plot([acc * 100 for acc in history['val_accuracy']], label='Validation Accuracy', color='green')
axes[1].axhline(y=max(history['val_accuracy']) * 100, color='red', linestyle='--', 
                label=f'Best: {max(history["val_accuracy"])*100:.1f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Interpretation:
# - If train loss keeps dropping but val loss increases = overfitting
# - If both plateau early = model capacity reached
# - If val accuracy plateaus around 55-60% = MLP ceiling for CIFAR-10

## Reflection

After completing the project, answer these questions:

1. What was the highest validation accuracy you achieved?

2. What hyperparameters worked best?

3. Where did the MLP struggle? (Which classes were confused?)

4. Why do you think CNNs would do better on this task?

5. What would you do differently next time?

---

*Updated: 2025_01_10_2145*

## Extended Learning Curves Analysis

Re-run training with additional metrics (train accuracy, learning rate) to get complete learning curve visualization.

In [ ]:
# Updated train_loop that also returns training accuracy
def train_loop_extended(dataloader, model, loss_fn, optimizer, device):
    """
    Training loop that tracks both loss AND accuracy.
    Returns: (average_loss, accuracy)
    """
    model.train()
    total_loss = 0
    correct = 0
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()  # Track correct predictions

    return total_loss / num_batches, correct / size


# Training with extended history tracking
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = OptimizedMLP().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

# Extended history - now includes train_accuracy and learning_rate
history = {
    'train_loss': [],
    'val_loss': [],
    'train_accuracy': [],    # NEW: track training accuracy
    'val_accuracy': [],
    'learning_rate': []      # NEW: track LR changes from scheduler
}

# Early stopping setup
best_val_acc = 0
patience = 15
patience_counter = 0
best_model_state = None

num_epochs = 100
for epoch in range(num_epochs):
    # Use extended train loop that returns accuracy
    train_loss, train_accuracy = train_loop_extended(train_dataloader, model, loss_fn, optimizer, device)
    val_loss, val_accuracy = test_loop(val_dataloader, model, loss_fn, device)

    # Store all metrics
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_accuracy'].append(train_accuracy)
    history['val_accuracy'].append(val_accuracy)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])  # Capture current LR
    
    scheduler.step(val_accuracy)
    
    # Early stopping
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}: Train Loss: {train_loss:.4f} | Train Acc: {100*train_accuracy:.1f}% | Val Acc: {100*val_accuracy:.1f}% | LR: {current_lr:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\n" + "="*60)
print("TRAINING COMPLETE - Extended history captured")
print("="*60)
print(f"Epochs run: {len(history['train_loss'])}")
print(f"Best val accuracy: {100*max(history['val_accuracy']):.1f}%")

In [ ]:
# =============================================================================
# COMPREHENSIVE LEARNING CURVES - 4 PLOTS
# =============================================================================
# This visualization helps diagnose training problems and understand model behavior.

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# =============================================================================
# PLOT 1: LOSS CURVES (Top Left)
# PURPOSE: Detect overfitting by comparing train vs validation loss
# 
# WHAT TO LOOK FOR:
#   - Both decreasing together = healthy learning
#   - Train ↓ while val ↑ = OVERFITTING (model memorizing, not generalizing)
#   - Large gap between curves = severe overfitting
#   - Both plateau = model capacity reached
# =============================================================================
axes[0, 0].plot(history['train_loss'], label='Training Loss', color='blue')
axes[0, 0].plot(history['val_loss'], label='Validation Loss', color='orange')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Loss Curves - Detect Overfitting')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# =============================================================================
# PLOT 2: ACCURACY CURVES (Top Right)
# PURPOSE: See actual performance and the "accuracy gap"
# 
# WHAT TO LOOK FOR:
#   - Train accuracy near 100% + val stuck = overfitting (memorization)
#   - Both improving together = healthy learning
#   - Val accuracy plateau at ~55-60% = MLP ceiling on CIFAR-10
#   - Gap between train and val = generalization gap
# =============================================================================
axes[0, 1].plot([acc * 100 for acc in history['train_accuracy']], label='Training Accuracy', color='blue', alpha=0.7)
axes[0, 1].plot([acc * 100 for acc in history['val_accuracy']], label='Validation Accuracy', color='green')
axes[0, 1].axhline(y=max(history['val_accuracy']) * 100, color='red', linestyle='--', 
                   label=f'Best Val: {max(history["val_accuracy"])*100:.1f}%')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Accuracy Curves - Training vs Validation Gap')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# =============================================================================
# PLOT 3: LEARNING RATE SCHEDULE (Bottom Left)
# PURPOSE: See when and how the scheduler reduced LR
# 
# WHAT TO LOOK FOR:
#   - Steps down = scheduler detected plateau, reduced LR
#   - Frequent drops = model struggling to improve
#   - High LR early = fast initial learning
#   - Low LR late = fine-tuning phase
#   - LR drops often correlate with accuracy improvements afterward
# =============================================================================
axes[1, 0].plot(history['learning_rate'], label='Learning Rate', color='purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_title('Learning Rate Schedule - When Did LR Reduce?')
axes[1, 0].set_yscale('log')  # Log scale - LR drops are exponential (0.01 → 0.005 → 0.0025...)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# =============================================================================
# PLOT 4: OVERFITTING GAP (Bottom Right)
# PURPOSE: Explicitly visualize the gap between train and val accuracy
# 
# WHAT TO LOOK FOR:
#   - Gap increasing = overfitting getting worse over time
#   - Gap stable = regularization (dropout, etc.) is working
#   - Gap > 20% = severe overfitting
#   - Gap near 0% = model might be underfitting (not learning enough)
# =============================================================================
accuracy_gap = [train - val for train, val in zip(history['train_accuracy'], history['val_accuracy'])]
axes[1, 1].plot([gap * 100 for gap in accuracy_gap], label='Train - Val Accuracy Gap', color='red', linewidth=2)
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1, 1].fill_between(range(len(accuracy_gap)), 0, [gap * 100 for gap in accuracy_gap], alpha=0.3, color='red')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy Gap (%)')
axes[1, 1].set_title('Overfitting Gap - How Much Better on Training Data?')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# =============================================================================
# INTERPRETATION SUMMARY
# =============================================================================
print("=" * 70)
print("INTERPRETATION SUMMARY")
print("=" * 70)
print(f"Final training accuracy:  {history['train_accuracy'][-1]*100:.1f}%")
print(f"Best validation accuracy: {max(history['val_accuracy'])*100:.1f}%")
print(f"Final accuracy gap:       {(history['train_accuracy'][-1] - history['val_accuracy'][-1])*100:.1f}%")
print(f"Learning rate reduced:    {history['learning_rate'][0]:.6f} → {history['learning_rate'][-1]:.6f}")
print("-" * 70)
print("DIAGNOSIS:")
if history['train_accuracy'][-1] > 0.95 and max(history['val_accuracy']) < 0.65:
    print("  → SEVERE OVERFITTING: Train acc near 100%, val stuck. Model memorized.")
elif (history['train_accuracy'][-1] - max(history['val_accuracy'])) > 0.20:
    print("  → MODERATE OVERFITTING: >20% gap. Consider more regularization.")
elif max(history['val_accuracy']) < 0.55:
    print("  → UNDERFITTING: Val acc below 55%. Try larger model or longer training.")
else:
    print("  → GOOD FIT: Reached expected MLP ceiling (~55-60%) with controlled gap.")
print("=" * 70)

### Confusion matrix

In [ ]:
  # 1. Collect predictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X, y in val_dataloader:
        X = X.to(device)
        preds = model(X).argmax(1).cpu()  # Get predicted class
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

In [ ]:

from sklearn.metrics import confusion_matrix
import seaborn as sns

# 1. Collect predictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X, y in val_dataloader:
        X = X.to(device)
        preds = model(X).argmax(1).cpu()  # Get predicted class
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

# 2. Create confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# 3. Visualize
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Per-class accuracy - which classes are easiest/hardest?
# Uses confusion matrix diagonal (correct predictions) divided by row sum (total per class)

per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

# Sort by accuracy (worst first) to see problem areas
sorted_indices = np.argsort(per_class_accuracy)

print("Per-Class Accuracy (sorted worst to best):")
print("-" * 45)
for idx in sorted_indices:
    accuracy = per_class_accuracy[idx] * 100
    print(f"{class_names[idx]:12s}: {accuracy:5.1f}%")

print("-" * 45)
print(f"Overall accuracy: {per_class_accuracy.mean()*100:.1f}%")

In [ ]:
# Visualize Predictions - Correct vs Misclassified
# See what the model gets right/wrong and WHY it might be confused

# Get predictions with confidence scores
model.eval()
all_images = []
all_preds = []
all_labels = []
all_confidences = []

with torch.no_grad():
    for X, y in val_dataloader:
        X = X.to(device)
        outputs = model(X)
        probs = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
        confidence, preds = probs.max(dim=1)   # Get highest probability and its class
        
        all_images.extend(X.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())
        all_confidences.extend(confidence.cpu().numpy())

all_images = np.array(all_images)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_confidences = np.array(all_confidences)

# Find correct and incorrect predictions
correct_mask = all_preds == all_labels
incorrect_mask = ~correct_mask

# Helper to display images
def show_predictions(indices, title, show_confidence=True):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        if i >= len(indices):
            ax.axis('off')
            continue
            
        idx = indices[i]
        image = all_images[idx].reshape(3, 32, 32).transpose(1, 2, 0)
        
        ax.imshow(image)
        
        true_label = class_names[all_labels[idx]]
        pred_label = class_names[all_preds[idx]]
        conf = all_confidences[idx] * 100
        
        if all_preds[idx] == all_labels[idx]:
            color = 'green'
            title_text = f'{pred_label}\n({conf:.0f}%)'
        else:
            color = 'red'
            title_text = f'Pred: {pred_label}\nTrue: {true_label}\n({conf:.0f}%)'
        
        ax.set_title(title_text, color=color, fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Show 10 correct predictions (high confidence)
correct_indices = np.where(correct_mask)[0]
high_conf_correct = correct_indices[np.argsort(all_confidences[correct_indices])[-10:]]
show_predictions(high_conf_correct, 'CORRECT Predictions (High Confidence)')

# Show 10 incorrect predictions (the model was confident but wrong)
incorrect_indices = np.where(incorrect_mask)[0]
high_conf_wrong = incorrect_indices[np.argsort(all_confidences[incorrect_indices])[-10:]]
show_predictions(high_conf_wrong, 'WRONG Predictions (Model Was Confident But Wrong)')

# Show cat↔dog confusions specifically
cat_idx, dog_idx = class_names.index('cat'), class_names.index('dog')
cat_as_dog = np.where((all_labels == cat_idx) & (all_preds == dog_idx))[0][:5]
dog_as_cat = np.where((all_labels == dog_idx) & (all_preds == cat_idx))[0][:5]
cat_dog_confusions = np.concatenate([cat_as_dog, dog_as_cat])
if len(cat_dog_confusions) > 0:
    show_predictions(cat_dog_confusions, 'Cat↔Dog Confusions - Why Does MLP Struggle?')

### Task 6.2: Final Test Set Evaluation

We've been using a validation split from the training data. CIFAR-10 has a separate held-out test set (`test_batch`) that we haven't touched. This gives us the "true" final accuracy.

In [ ]:
# Load the held-out test set (never seen during training or validation)
with open('/opt/ml-datasets/cifar10/cifar-10-batches-py/test_batch', 'rb') as file:
    test_batch = pickle.load(file, encoding='bytes')

test_data = test_batch[b'data'].astype(np.float32) / 255.0
test_labels = np.array(test_batch[b'labels'])

print(f"Test set: {test_data.shape[0]} images")

# Convert to tensors and create dataloader
test_data_tensor = torch.tensor(test_data, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)
test_dataset = TensorDataset(test_data_tensor, test_labels_tensor)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Evaluate on test set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        predictions = outputs.argmax(1)
        correct += (predictions == y).sum().item()
        total += y.size(0)

test_accuracy = correct / total

print("=" * 50)
print("FINAL TEST SET RESULTS")
print("=" * 50)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Validation Accuracy (best): {max(history['val_accuracy']) * 100:.2f}%")
print("-" * 50)
print(f"Difference: {(max(history['val_accuracy']) - test_accuracy) * 100:+.2f}%")
print("=" * 50)

# Small difference = good generalization
# Large difference = overfitted to validation set (rare but possible)

### Task 6.6: Understanding MLP Limitations

Based on our analysis, here's what we learned about MLPs on image classification:

---

#### What the MLP Learned (Color Statistics)

The MLP sees images as **flat vectors of 3072 numbers**. It has no concept of:
- Pixels being neighbors
- Edges, shapes, or contours
- Objects being in different positions

Instead, it learns **color/texture correlations**:
- "Lots of blue pixels → probably sky → airplane or bird"
- "Brown/tan colors → probably animal → cat, dog, deer, horse"
- "Metallic gray + red → probably vehicle → truck or automobile"

---

#### Why Certain Classes Get Confused

| Confusion | Why |
|-----------|-----|
| **Cat ↔ Dog** | Similar fur colors/textures, similar sizes in frame |
| **Bird ↔ Airplane** | Both often on blue sky backgrounds |
| **Bird ↔ Deer** | Both have natural/outdoor color palettes |
| **Truck ↔ Automobile** | Similar metallic colors, road backgrounds |

---

#### The Fundamental Problem

When you **move a dog from the left to the right** of an image:
- To a human: same dog, same class
- To an MLP: completely different input vector → might predict differently

This is called **lack of translation invariance**. MLPs treat `[pixel_0, pixel_1, ...]` as independent features with no spatial relationship.

---

#### MLP Ceiling vs CNN Performance

| Architecture | CIFAR-10 Accuracy | Why |
|--------------|-------------------|-----|
| **MLP (ours)** | ~55-60% | Learns color statistics only |
| **Simple CNN** | ~75-85% | Learns local patterns (edges, textures) |
| **Deep CNN (ResNet)** | ~93-96% | Hierarchical features (edges → shapes → objects) |

The **30%+ gap** between MLP and CNN shows that **architecture matters more than hyperparameter tuning**.

---

#### Key Takeaway

> We spent significant effort tuning our MLP (dropout, BatchNorm, LR scheduling, early stopping) to go from 53% → 59%. A basic CNN would hit 75%+ with minimal tuning.
>
> **Lesson:** Choose the right architecture for the problem before optimizing hyperparameters.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>